<a href="https://colab.research.google.com/github/arnauatgit/AIMD/blob/main/Notebooks/Problemes2.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/>
</a>

# Problemes 2

**Part 2.1. Introducció**

In [ ]:
# connectar el google drive
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
# definicio directoris
from pathlib import Path
import sys

AIMD_ROOT  = '/content/drive/MyDrive/AIMD'
DATA_DIR    = AIMD_ROOT + '/Data'
MRBRAINS_DIR = DATA_DIR + '/MRBrainS'
RESULTS_DIR = AIMD_ROOT + '/Results'

sys.path.append(str(AIMD_ROOT))

In [ ]:
# Instal·lació de les llibreries necessàries
%pip install -q antspyx antspynet

**Part 2.2. ITK-Snap**

Visualitza les imatges de treball utilitzant el programa ITK-Snap (https://www.itksnap.org/pmwiki/pmwiki.php?n=Downloads.SNAP4)

**Part 2.3. Pre-procés**

Començarem la part d'implementació pre-processant les imatges. Ho farem amb la llibreria AntPy i AntsPyNet, que permeten treure el bias field i realitzar l'skull stripping automàticament.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ants, antspynet

# llegir la imatge
ants_img = ants.image_read(MRBRAINS_DIR+"/1/T1.nii")

# N4 directe amb ANTs
img_n4 = ants.n4_bias_field_correction(ants_img)

# quedar-nos amb el cervell
brain = antspynet.brain_extraction(img_n4, modality="t1")
brain_mask = ants.threshold_image(brain, 0.5, 1.0, 1, 0)
brain_extracted = img_n4 * brain_mask

# passar a numpy
img_np = img_n4.numpy()
brain_extracted_np = brain_extracted.numpy()

# buscar el tall central
z = img_np.shape[2] // 2
llesca_img = img_np[:,:,z]
llesca_extracted = brain_extracted_np[:,:,z]

# mostrar les imatges
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.imshow(llesca_img, cmap='gray')
plt.title("Original")
plt.axis('off')
plt.subplot(1,2,2)
plt.imshow(llesca_extracted, cmap='gray')
plt.title("Skull stripped")
plt.axis('off')
plt.show()

Exercici 2.1. Explica el codi per fer l'skull stripping. Què fa cada funció?
```text
brain = antspynet.brain_extraction(img_n4, modality="t1")
brain_mask = ants.threshold_image(brain, 0.5, 1.0, 1, 0)
brain_extracted = img_n4 * brain_mask
```

Exercici 2.2. Aplica aquest preprocessament a una imatge MRI del dataset de treball i compara-les visualment. Analitza especialment els aspectes següents:
- La reducció de variacions d'intensitat dins del cervell.
- L'eliminació d'estructures no cerebrals (crani i fons).
- Els canvis en el contrast entre substància grisa, substància blanca i CSF.

**Part 2.4. kMeans 2D**

Exercici 2.3. Extreu una llesca central del volum MRI preprocessat (preferentment en el pla axial), i visualitza-la. Aplica k-Means utilitzant 4 clústers. En aquest cas, estem assuming que tres clusters corresponen als teixits cerebrals (substància grisa, substància blanca i CSF) mentre que el quart correspon al fons. Com veus el resultat?

In [ ]:
from sklearn.cluster import KMeans

# Estirar la imatge
B = llesca_extracted.reshape(-1,1)


Exercici 2.4. De fet, però, ja tenim ben segmentat el fons. La millor manera és obviar la seva presència en els càlculs del k-Means. Això es coneix com segmentar amb màscara. Utilitza la màscara per emmascarar només els píxels útils necessaris per segmentar uilitzant 3 classes.

Compara quantitativament la segmentació amb i sense màscara, i analitza si la eliminació del fons afecta la qualitat i estabilitat dels clusters.  Implementa la funció Dice al fitxer de les nostres mètriques.

In [ ]:
# Agafar només valors més grans que 0
B = llesca_extracted.reshape(-1,1)
C = B[B>0].reshape(-1,1)


In [ ]:
from Utils.metrics import Dice



**Part 2.5. kMeans 2.5D**

Exercici 2.5. Segmentar llesca a llesca i recuperar volum

In [ ]:
def segmenta(llesca):

# segmentar
brain_seg = np.zeros(brain_extracted.shape)
for i in range(brain_extracted.shape[2]):
  llesca = brain_extracted_np[:,:,i]
  llesca_seg = segmenta(llesca)
  brain_seg[:,:,i] = llesca_seg

# visualitzar
fig, ax = plt.subplots(6, 8, figsize=(12, 9))
for i in range(brain_extracted.shape[2]):
    ax[i//8, i%8].imshow(brain_seg[:,:,i], cmap='gray')
    ax[i//8, i%8].set_title(f"{i}")
    ax[i//8, i%8].axis('off')

plt.tight_layout()
plt.show()


In [ ]:
# guardar la imatge
seg_img = ants_img.new_image_like(brain_seg.astype(np.uint8))
ants.image_write(seg_img, RESULTS_DIR+"/brain_seg25d.nii.gz")

**Part 2.6. kMeans 3D**

In [ ]:
# Agafar només valors més grans que 0
B = brain_extracted_np.reshape(-1,1)
C = B[B>0].reshape(-1,1)


In [ ]:
# guardar la imatge
seg_img = ants_img.new_image_like(B_seg.astype(np.uint8))
ants.image_write(seg_img, RESULTS_DIR+"/brain_seg3d.nii.gz")


**Part 2.7. Avaluació en múltiples subjectes**

Exercici 2.7. Segmenta tots els subjectes del dataset MRBrainS per avaluar la robustesa del mètode. Fes un boxplot per sumaritzar els resultats.

In [ ]:
def PreprocessatAnts(imatge):


In [ ]:
def SegmentaBrain(imatge):


In [ ]:
def AvaluaSeg(seg1,seg2):


In [ ]:
wm = np.zeros(5); gm = np.zeros(5); csf = np.zeros(5)
for i, pacient in enumerate(range(1, 6)):
    filename = f"{MRBRAINS_DIR}/{pacient}/T1.nii"
    gt_filename = f"{MRBRAINS_DIR}/{pacient}/LabelsForTesting.nii"

    # Processar
    print(filename)

    # Script que llegeix la imatge i el ground-truth corresponent,
    # preprocessa i segmenta la imatge i compara el resultat amb el grount-truth



    # guardar la imatge
    seg_img = ants_img.new_image_like(brain_seg.astype(np.uint8))
    ants.image_write(seg_img, f"{RESULTS_DIR}/{pacient}_seg.nii.gz")

print(f"Dice WM: {wm.mean():.2f}")
print(f"Dice GM: {gm.mean():.2f}")
print(f"Dice CSF: {csf.mean():.2f}")
plt.boxplot([wm,gm,csf])
plt.xticks([1,2,3], ['WM','GM','CSF'])
plt.show()
